In [44]:
import numpy as np
import pyarrow as pa
import pandas as pd
import pyarrow.parquet as pq
import duckdb
from pathlib import Path
import time
import pyarrow.compute as pc

In [9]:
np.random.seed(42)
n=500000 #rows
cities = ['Baku','Honolulu','Istanbul','Budapest','Athens','Moscow','Los-Angeles','Toronto','Dubai','Rio de Janeiro']
end_date=pd.Timestamp.now()
start_date=end_date-pd.Timedelta(days=1095)
random_days=np.random.randint(0,1095,n)
random_seconds=np.random.randint(0,86400,n)
random_timedeltas=pd.to_timedelta(random_days,unit='D')+pd.to_timedelta(random_seconds,unit='s')

df = pd.DataFrame({
    'user_id':np.arange(1,n+1),
    'city':np.random.choice(cities,n),
    'score':np.round(np.random.uniform(0,100,n),2),
    'active':np.random.choice([True,False],n),   
    'signup_date':start_date+random_timedeltas,
    'age':np.random.randint(18,81,n),
    'sessions':np.random.randint(0,501,n),
    'revenue':np.round(np.random.uniform(0,1001,n),2)
    })
df.shape

(500000, 8)

In [24]:
df.to_parquet('dataset.parquet',index=False,engine='pyarrow')

In [25]:
parquet_file = pq.ParquetFile("dataset")
print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")
print(f"Number of rows: {parquet_file.metadata.num_rows}")
print(f"\nSchema:")
print(parquet_file.schema_arrow)

Number of row groups: 1
Number of columns: 8
Number of rows: 500000

Schema:
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 974


In [26]:
for name, pa_type in zip(parquet_file.schema_arrow.names, parquet_file.schema_arrow.types):
    print(f"- {name}: {pa_type}")
# --- First Row Group Statistics ---
print("\n### Column Statistics (First Row Group) ###")
# Get metadata for the very first row group
first_row_group = parquet_file.metadata.row_group(0)

for i in range(first_row_group.num_columns):
    col_meta = first_row_group.column(i)
    
    col_name = col_meta.path_in_schema
    physical_type = col_meta.physical_type
    compressed_size = col_meta.total_compressed_size
    
    print(f"\n**{col_name}**")
    print(f"  Physical Type: {physical_type}")
    print(f"  Compressed Size: {compressed_size} bytes")
    
    # Extract statistics if they exist
    stats = col_meta.statistics
    if stats and stats.has_min_max:
        print(f"  Min: {stats.min}")
        print(f"  Max: {stats.max}")
        print(f"  Null Count: {stats.null_count}")
    else:
        print("  Statistics: Not available/Not computed")

- user_id: int64
- city: string
- score: double
- active: bool
- signup_date: timestamp[ns]
- age: int32
- sessions: int32
- revenue: double

### Column Statistics (First Row Group) ###

**user_id**
  Physical Type: INT64
  Compressed Size: 2280081 bytes
  Min: 1
  Max: 500000
  Null Count: 0

**city**
  Physical Type: BYTE_ARRAY
  Compressed Size: 251194 bytes
  Min: Athens
  Max: Toronto
  Null Count: 0

**score**
  Physical Type: DOUBLE
  Compressed Size: 923132 bytes
  Min: -0.0
  Max: 100.0
  Null Count: 0

**active**
  Physical Type: BOOLEAN
  Compressed Size: 62553 bytes
  Min: False
  Max: True
  Null Count: 0

**signup_date**
  Physical Type: INT64
  Compressed Size: 4297269 bytes
  Min: 2023-03-05 00:34:54.681244
  Max: 2026-03-04 00:29:35.681244
  Null Count: 0

**age**
  Physical Type: INT32
  Compressed Size: 376346 bytes
  Min: 18
  Max: 80
  Null Count: 0

**sessions**
  Physical Type: INT32
  Compressed Size: 565608 bytes
  Min: 0
  Max: 500
  Null Count: 0

**revenue**

In [34]:
df.to_csv('dataset.csv',index=False)
csv_size=(Path('dataset.csv').stat().st_size)/1024
pq_size=(Path('dataset.parquet').stat().st_size)/1024
compression_ratio=csv_size/pq_size
print(f"Size: {csv_size} Kbytes")
print(f"Size: {pq_size} Kbytes")
print(f'compression ratio is {compression_ratio}')


Size: 33375.734375 Kbytes
Size: 10062.109375 Kbytes
compression ratio is 3.316971932140223


parquet provides statistics for each chunk and that reduces time of analysis on data, for example, if we want to get signup date>something, system checks every chunks maximums, and if this maximum is lower than desired signup date, these chunks does not get read or analysed.

In [36]:
parquet_path = 'dataset.parquet'
csv_path = 'dataset.csv'
columns_to_read = ['user_id', 'city']

start_time = time.perf_counter()
df_parquet_full = pd.read_parquet(parquet_path)
parquet_full_time = time.perf_counter() - start_time

print(f"Step 1 | Read Full Parquet: {parquet_full_time:.4f} seconds")

start_time = time.perf_counter()
df_parquet_subset = pd.read_parquet(parquet_path, columns=columns_to_read)
parquet_subset_time = time.perf_counter() - start_time

print(f"Step 2 | Read 2 Columns (Parquet): {parquet_subset_time:.4f} seconds")
start_time = time.perf_counter()
df_csv_full = pd.read_csv(csv_path)
df_csv_subset = df_csv_full[columns_to_read]
csv_time = time.perf_counter() - start_time

print(f"Step 3 | Read 2 Columns (CSV): {csv_time:.4f} seconds")

print(f"Parquet subset was {csv_time / parquet_subset_time:.2f}x faster than CSV subset.")

Step 1 | Read Full Parquet: 0.1364 seconds
Step 2 | Read 2 Columns (Parquet): 0.0289 seconds
Step 3 | Read 2 Columns (CSV): 0.3132 seconds
Parquet subset was 10.83x faster than CSV subset.


Parquet is faster for selective reads because it stores data by column instead of by row. In a CSV file, the data is laid out row by row. If you only want to read one column, the engine still has to read through the entire file on the disk to find the pieces you want, which takes a lot of time and disk activity.

Parquet packs all the data for a single column tightly together. When you tell it to read just one or two columns, it can jump straight to those specific blocks on the disk and completely ignore the rest of the file. Because it physically reads way less data from the hard drive, it finishes the job much faster.

In [45]:
file_path = 'dataset.parquet'

# --- Step 1: PyArrow with Filter (Predicate Pushdown) ---
start = time.perf_counter()
# The 'filters' argument is pushed down to the Parquet reader
table_arrow = pq.read_table(file_path, filters=(pc.field("age") > 50))
df_arrow = table_arrow.to_pandas()
time_arrow = time.perf_counter() - start

# --- Step 2: Pandas Full Read + Post-Filter ---
start = time.perf_counter()
df_pandas_full = pd.read_parquet(file_path)
df_pandas_filtered = df_pandas_full[df_pandas_full['age'] > 50]
time_pandas = time.perf_counter() - start

# --- Step 4: DuckDB SQL Interface ---
start = time.perf_counter()
# DuckDB scans the Parquet file directly using its highly optimized engine
df_duck = duckdb.sql(f"SELECT * FROM '{file_path}' WHERE age > 50").df()
time_duck = time.perf_counter() - start

# --- Step 3: Comparison & Results ---
print(f"PyArrow Filtered Rows: {len(df_arrow):,}")
print(f"Pandas Filtered Rows:  {len(df_pandas_filtered):,}")
print(f"DuckDB Filtered Rows:  {len(df_duck):,}")

print(f"\nExecution Times:")
print(f"PyArrow (Pushdown):    {time_arrow:.4f}s")
print(f"Pandas (Full Read):    {time_pandas:.4f}s")
print(f"DuckDB (SQL Scan):     {time_duck:.4f}s")

PyArrow Filtered Rows: 238,701
Pandas Filtered Rows:  238,701
DuckDB Filtered Rows:  238,701

Execution Times:
PyArrow (Pushdown):    0.0520s
Pandas (Full Read):    0.0459s
DuckDB (SQL Scan):     0.0458s


In [46]:
df = pd.read_parquet('dataset.parquet')
file_path = 'dataset.parquet'
con = duckdb.connect()

def benchmark(label, func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.4f}s")
    return elapsed

results = {}

#Query 1
results['Q1_pandas'] = benchmark("Q1 Pandas", lambda: df['city'].value_counts())
results['Q1_duckdb'] = benchmark("Q1 DuckDB", lambda: con.sql(f"SELECT city, COUNT(*) FROM '{file_path}' GROUP BY city").df())

#Query 2
results['Q2_pandas'] = benchmark("Q2 Pandas", lambda: df.groupby('city')['score'].mean().sort_values(ascending=False))
results['Q2_duckdb'] = benchmark("Q2 DuckDB", lambda: con.sql(f"SELECT city, AVG(score) as avg_score FROM '{file_path}' GROUP BY city ORDER BY avg_score DESC").df())

#Query 3
def q3_pd():
    active_high = df[(df['active'] == True) & (df['score'] > 75)]
    counts = active_high.groupby('city').size()
    total_active = df[df['active'] == True].groupby('city').size()
    return (counts / total_active) * 100

results['Q3_pandas'] = benchmark("Q3 Pandas", q3_pd)
results['Q3_duckdb'] = benchmark("Q3 DuckDB", lambda: con.sql(f"""
    SELECT city, 
           (COUNT(CASE WHEN score > 75 THEN 1 END) * 100.0 / COUNT(*)) as pct
    FROM '{file_path}' 
    WHERE active = True 
    GROUP BY city
""").df())

#Query 4
results['Q4_pandas'] = benchmark("Q4 Pandas", lambda: df[df.groupby('city')['score'].rank(method='first', ascending=False) <= 10])
results['Q4_duckdb'] = benchmark("Q4 DuckDB", lambda: con.sql(f"""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER(PARTITION BY city ORDER BY score DESC) as rank
        FROM '{file_path}'
    ) WHERE rank <= 10
""").df())

#Query 5
results['Q5_pandas'] = benchmark("Q5 Pandas", lambda: df.sort_values(['city', 'user_id']).groupby('city')['score'].cumsum())
results['Q5_duckdb'] = benchmark("Q5 DuckDB", lambda: con.sql(f"""
    SELECT city, user_id, score, 
           SUM(score) OVER(PARTITION BY city ORDER BY user_id) as running_total
    FROM '{file_path}'
""").df())

Q1 Pandas: 0.0129s
Q1 DuckDB: 0.0045s
Q2 Pandas: 0.0184s
Q2 DuckDB: 0.0053s
Q3 Pandas: 0.0224s
Q3 DuckDB: 0.0104s
Q4 Pandas: 0.1258s
Q4 DuckDB: 0.2015s
Q5 Pandas: 0.0822s
Q5 DuckDB: 0.1197s


writing queries is easier in duckDB. first 3 queries was faster in duckDB, forth and fifth was faster in pandas. in first and second queries, ratio was the largest

In [ ]:
arrow_table = pa.Table.from_pandas(df)

print(f"{'Column':<15} | {'Pandas Dtype':<15} | {'Arrow Type':<15}")
print("-" * 50)
for col in df.columns:
    p_type = str(df[col].dtype)
    a_type = str(arrow_table.schema.field(col).type)
    print(f"{col:<15} | {p_type:<15} | {a_type:<15}")

pq.write_table(arrow_table, 'arrow_bridge.parquet')
arrow_table_back = pq.read_table('arrow_bridge.parquet')

df_back = arrow_table_back.to_pandas()

matches = df.equals(df_back)
print(f"Step 4: Converted back to Pandas. Data matches original: {matches}")

df_arrow_backed = pd.read_parquet('arrow_bridge.parquet', engine='pyarrow', dtype_backend="pyarrow")

print(f"{'Column':<15} | {'Traditional Dtype':<20} | {'Arrow-backed Dtype':<20}")
print("-" * 65)
for col in df.columns:
    trad = str(df[col].dtype)
    backed = str(df_arrow_backed[col].dtype)
    print(f"{col:<15} | {trad:<20} | {backed:<20}")

Step 1: Successfully converted Pandas DataFrame to Arrow Table.

--- Step 2: Schema Comparison ---
Column          | Pandas Dtype    | Arrow Type     
--------------------------------------------------
user_id         | int64           | int64          
city            | object          | string         
score           | float64         | double         
active          | bool            | bool           
signup_date     | datetime64[ns]  | timestamp[ns]  
age             | int32           | int32          
sessions        | int32           | int32          
revenue         | float64         | double         

Step 3: Wrote to Parquet and read back into Arrow Table.
Step 4: Converted back to Pandas. Data matches original: True
Column          | Traditional Dtype    | Arrow-backed Dtype  
-----------------------------------------------------------------
user_id         | int64                | int64[pyarrow]      
city            | object               | string[pyarrow]     
score     

Apache Arrow is a common language for data that lets different tools work together without slow translations. Parquet stores data in columns on disk, while Arrow keeps it in columns in RAM. cause they use the same layout, data moves from disk to memory almost instantly. DuckDB uses this to run SQL queries at high speed, and pandas uses it to handle data without the overhead of type conversion. Arrow acts as bridge that connects storage,SQL engine, and Python analysis into one system.